# Entregable BI iData Global – Olist con DuckClaw

Prueba técnica: análisis de comportamiento de clientes y tendencias de ventas usando **DuckClaw** (DuckDB), consultas en lenguaje natural (Groq) y **gráficas** con matplotlib/seaborn.

## 1. Configuración y carga de datos

Creamos la base DuckClaw y cargamos los CSV de Olist desde `data/`. Ejecuta estas celdas primero.

In [1]:
import json
import duckclaw
from pathlib import Path

# Ruta al repo: desde repo root (data/) o desde notebooks/ (../data/)
REPO_ROOT = Path.cwd() if (Path.cwd() / "data" / "olist_orders_dataset.csv").exists() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "data"
DB_PATH = str(REPO_ROOT / "olist_bi.duckdb")

db = duckclaw.DuckClaw(str(DB_PATH))

In [2]:
print(db)

In [3]:
from duckclaw.bi import load_olist_data

counts = load_olist_data(db, str(DATA_DIR))
print("Tablas cargadas:")
for table, n in counts.items():
    print(f"  {table}: {n:,} filas")

Tablas cargadas:
  olist_orders: 99,441 filas
  olist_customers: 99,441 filas
  olist_order_items: 112,650 filas
  olist_order_payments: 103,886 filas
  olist_order_reviews: 99,224 filas
  olist_products: 32,951 filas
  olist_sellers: 3,095 filas
  product_category_name_translation: 71 filas
  olist_geolocation: 1,000,163 filas


In [4]:
import importlib
import duckclaw.bi.olist
import duckclaw.bi.agent
importlib.reload(duckclaw.bi.olist)
importlib.reload(duckclaw.bi.agent)

<module 'duckclaw.bi.agent' from '/Users/juanjosearevalocamargo/Desktop/duckclaw/duckclaw/bi/agent.py'>

In [5]:
from pathlib import Path
import duckclaw
from duckclaw.bi import load_olist_data, ask_bi

REPO_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "data"

db = duckclaw.DuckClaw(str(REPO_ROOT / "olist_bi.duckdb"))
load_olist_data(db, str(DATA_DIR))
counts = load_olist_data(db, str(DATA_DIR))
print(counts)

{'olist_orders': 99441, 'olist_customers': 99441, 'olist_order_items': 112650, 'olist_order_payments': 103886, 'olist_order_reviews': 99224, 'olist_products': 32951, 'olist_sellers': 3095, 'product_category_name_translation': 71, 'olist_geolocation': 1000163}


## 2. Preguntas de negocio obligatorias

### 2.1 Clientes que más generan ventas

In [6]:
from pathlib import Path
from duckclaw.bi import load_olist_data, ask_bi
"""
import json
import duckclaw

REPO_ROOT = Path.cwd() if (Path.cwd() / "data" / "olist_orders_dataset.csv").exists() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "data"
DB_PATH = str(REPO_ROOT / "olist_bi.duckdb")

db = duckclaw.DuckClaw(DB_PATH)
load_olist_data(db, str(DATA_DIR))
"""

respuesta = ask_bi(
    db,
    "¿Quiénes son los mejores vendedores y cuál es el tiempo medio de entrega?",
    provider="groq",
)
print(respuesta)

/Users/juanjosearevalocamargo/Desktop/duckclaw/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Los mejores vendedores son:
1. El vendedor con id "4869f7a5dfa277a7dca6462dcf3b52b2" de la ciudad de Guariba, con un total de ventas de $247,007.06 y 1,124 pedidos.
2. El vendedor con id "7c67e1448b00f6e969d365cea6b010ab" de la ciudad de Itaquaquecetuba, con un total de ventas de $237,806.69 y 973 pedidos.
3. El vendedor con id "4a3ca9315b744ce9f8e9374361493884" de la ciudad de Ibitinga, con un total de ventas de $231,220.43 y 1,772 pedidos.

El tiempo medio de entrega es de 12.50 días. El tiempo mínimo de entrega es de 0 días y el máximo es de 210 días.


In [7]:
"""
from duckclaw.bi import ask_bi

db = duckclaw.DuckClaw("olist_bi.duckdb")
load_olist_data(db, "data")

respuesta = ask_bi(
    db,
    "¿Quiénes son los mejores vendedores y cuál es el tiempo medio de entrega?",
    provider="groq"
)
print(respuesta)
"""

'\nfrom duckclaw.bi import ask_bi\n\ndb = duckclaw.DuckClaw("olist_bi.duckdb")\nload_olist_data(db, "data")\n\nrespuesta = ask_bi(\n    db,\n    "¿Quiénes son los mejores vendedores y cuál es el tiempo medio de entrega?",\n    provider="groq"\n)\nprint(respuesta)\n'

In [8]:
from duckclaw.bi import get_top_customers_by_sales

top_customers = get_top_customers_by_sales(db, limit=15)
print(json.dumps(top_customers, indent=2, ensure_ascii=False, default=str))

[
  {
    "customer_id": "1617b1357756262bfa56ab541c47bc16",
    "customer_city": "rio de janeiro",
    "customer_state": "RJ",
    "total_sales": "13664.08",
    "num_orders": "1"
  },
  {
    "customer_id": "ec5b2ba62e574342386871631fafd3fc",
    "customer_city": "vila velha",
    "customer_state": "ES",
    "total_sales": "7274.88",
    "num_orders": "1"
  },
  {
    "customer_id": "c6e2731c5b391845f6800c97401a43a9",
    "customer_city": "campo grande",
    "customer_state": "MS",
    "total_sales": "6929.31",
    "num_orders": "1"
  },
  {
    "customer_id": "f48d464a0baaea338cb25f816991ab1f",
    "customer_city": "vitoria",
    "customer_state": "ES",
    "total_sales": "6922.21",
    "num_orders": "1"
  },
  {
    "customer_id": "3fd6777bbce08a352fddd04e4a7cc8f6",
    "customer_city": "marilia",
    "customer_state": "SP",
    "total_sales": "6726.66",
    "num_orders": "1"
  },
  {
    "customer_id": "05455dfa7cd02f13d132aa7a6a9729c6",
    "customer_city": "divinopolis",
    "cu

### 2.2 ¿A qué clientes debo fidelizar?

In [9]:
from duckclaw.bi import get_customers_to_retain

retain = get_customers_to_retain(db, limit=15, min_orders=2)
print(json.dumps(retain, indent=2, ensure_ascii=False, default=str))

[]


### 2.3 ¿Cuáles son mis mejores vendedores?

In [10]:
from duckclaw.bi import get_top_sellers

top_sellers = get_top_sellers(db, limit=15)
print(json.dumps(top_sellers, indent=2, ensure_ascii=False, default=str))

[
  {
    "seller_id": "4869f7a5dfa277a7dca6462dcf3b52b2",
    "seller_city": "guariba",
    "seller_state": "SP",
    "total_sales": "247007.06000000006",
    "num_orders": "1124"
  },
  {
    "seller_id": "7c67e1448b00f6e969d365cea6b010ab",
    "seller_city": "itaquaquecetuba",
    "seller_state": "SP",
    "total_sales": "237806.68999999983",
    "num_orders": "973"
  },
  {
    "seller_id": "4a3ca9315b744ce9f8e9374361493884",
    "seller_city": "ibitinga",
    "seller_state": "SP",
    "total_sales": "231220.43000000055",
    "num_orders": "1772"
  },
  {
    "seller_id": "53243585a1d6dc2643021fd1853d8905",
    "seller_city": "lauro de freitas",
    "seller_state": "BA",
    "total_sales": "230797.0200000003",
    "num_orders": "348"
  },
  {
    "seller_id": "fa1c13f2614d7b5c4749cbc52fecda94",
    "seller_city": "sumare",
    "seller_state": "SP",
    "total_sales": "200833.4999999998",
    "num_orders": "578"
  },
  {
    "seller_id": "da8622b14eb17ae2831f4ac5b9dab84a",
    "sell

### 2.4 ¿Cuál es el promedio de tiempos de entrega? ¿Hay casos críticos?

In [11]:
from duckclaw.bi import get_delivery_metrics, get_delivery_critical_cases

metrics = get_delivery_metrics(db)
print("Métricas de entrega:", json.dumps(metrics, indent=2, default=str))

critical = get_delivery_critical_cases(db, days_threshold=20, limit=10)
print("\nCasos críticos (>20 días):", json.dumps(critical, indent=2, default=str))

Métricas de entrega: [
  {
    "delivered_orders": "96470",
    "avg_delivery_days": "12.50",
    "min_days": "0.00",
    "max_days": "210.00"
  }
]

Casos críticos (>20 días): [
  {
    "order_id": "ca07593549f1816d26a572e06dc1eab6",
    "order_purchase_timestamp": "2017-02-21 23:31:27",
    "order_delivered_customer_date": "2017-09-19 14:36:39",
    "delivery_days": "210.00"
  },
  {
    "order_id": "1b3190b2dfa9d789e1f14c05b647a14a",
    "order_purchase_timestamp": "2018-02-23 14:57:35",
    "order_delivered_customer_date": "2018-09-19 23:24:07",
    "delivery_days": "208.00"
  },
  {
    "order_id": "440d0d17af552815d15a9e41abe49359",
    "order_purchase_timestamp": "2017-03-07 23:59:51",
    "order_delivered_customer_date": "2017-09-19 15:12:50",
    "delivery_days": "196.00"
  },
  {
    "order_id": "2fb597c2f772eca01b1f5c561bf6cc7b",
    "order_purchase_timestamp": "2017-03-08 18:09:02",
    "order_delivered_customer_date": "2017-09-19 14:33:17",
    "delivery_days": "195.00"
  

## 3. Preguntas adicionales de negocio

### Resumen de ventas (ticket, total)

In [12]:
from duckclaw.bi import get_sales_summary

summary = get_sales_summary(db)
print(json.dumps(summary, indent=2, default=str))

[
  {
    "total_orders": "96478",
    "total_sales": "15419773.75",
    "avg_ticket": "159.83"
  }
]


### Satisfacción (reviews)

In [13]:
from duckclaw.bi import get_review_metrics

reviews = get_review_metrics(db)
print(json.dumps(reviews, indent=2, default=str))

[
  {
    "avg_score": "4.09",
    "total_reviews": "99224",
    "good_reviews": "76470",
    "bad_reviews": "14575"
  }
]


### Ventas por categoría de producto

In [14]:
from duckclaw.bi import get_category_sales

categories = get_category_sales(db, limit=15)
print(json.dumps(categories, indent=2, ensure_ascii=False, default=str))

[
  {
    "category": "health_beauty",
    "total_sales": "1412089.53",
    "num_orders": "8647"
  },
  {
    "category": "watches_gifts",
    "total_sales": "1264333.12",
    "num_orders": "5495"
  },
  {
    "category": "bed_bath_table",
    "total_sales": "1225209.26",
    "num_orders": "9272"
  },
  {
    "category": "sports_leisure",
    "total_sales": "1118256.91",
    "num_orders": "7530"
  },
  {
    "category": "computers_accessories",
    "total_sales": "1032723.77",
    "num_orders": "6530"
  },
  {
    "category": "furniture_decor",
    "total_sales": "880329.92",
    "num_orders": "6307"
  },
  {
    "category": "housewares",
    "total_sales": "758392.25",
    "num_orders": "5743"
  },
  {
    "category": "cool_stuff",
    "total_sales": "691680.89",
    "num_orders": "3559"
  },
  {
    "category": "auto",
    "total_sales": "669454.75",
    "num_orders": "3810"
  },
  {
    "category": "garden_tools",
    "total_sales": "567145.68",
    "num_orders": "3448"
  },
  {
   

## 4. Uso opcional con pandas

Si tienes `pandas`, puedes convertir los resultados a DataFrame para tablas y gráficos.

In [15]:
try:
    import pandas as pd
    df_top = pd.DataFrame(get_top_customers_by_sales(db, limit=10))
    display(df_top)
except Exception as e:
    print("Sin pandas o display:", e)

Sin pandas o display: No module named 'pandas'


In [22]:
# Chatbot: escribe preguntas y recibe respuestas. "salir" o "exit" para terminar.
from duckclaw.bi import ask_bi

print("Chat BI Olist (Groq). Escribe tu pregunta o 'salir' para terminar.\n")
while True:
    pregunta = input("Tú: ").strip()
    if not pregunta:
        continue
    if pregunta.lower() in ("salir", "exit", "quit", "q"):
        print("Hasta luego.")
        break
    print("BI: ", end="")
    respuesta = ask_bi(db, pregunta, provider="groq")
    print(respuesta)
    print()

Chat BI Olist (Groq). Escribe tu pregunta o 'salir' para terminar.

BI: La tabla con más registros es la de pedidos (orders), con un total de 96,478 registros.

Hasta luego.


## 5. Preguntas en lenguaje natural con LLM (Groq)

Un LLM interpreta tu pregunta y usa las funciones DuckClaw como herramientas. También puedes **pedir gráficas** en texto (ej. *"gráfico de ventas por categoría"*). Requiere `GROQ_API_KEY` y `pip install -e ".[groq]"`.

In [17]:
from duckclaw.bi import ask_bi

# Pregunta en lenguaje natural; el LLM elige la herramienta y resume el resultado
respuesta = ask_bi(db, "¿Cuáles son los clientes que más ventas generan? Dame un resumen.", provider="groq")
print(respuesta)

Los clientes que más ventas generan son:

1. 1617b1357756262bfa56ab541c47bc16 con un total de ventas de $13,664.08 y 1 orden.
2. ec5b2ba62e574342386871631fafd3fc con un total de ventas de $7,274.88 y 1 orden.
3. c6e2731c5b391845f6800c97401a43a9 con un total de ventas de $6,929.31 y 1 orden.
4. f48d464a0baaea338cb25f816991ab1f con un total de ventas de $6,922.21 y 1 orden.
5. 3fd6777bbce08a352fddd04e4a7cc8f6 con un total de ventas de $6,726.66 y 1 orden.
6. 05455dfa7cd02f13d132aa7a6a9729c6 con un total de ventas de $6,081.54 y 1 orden.
7. df55c14d1476a9a3467f131269c2477f con un total de ventas de $4,950.34 y 1 orden.
8. 24bbf5fd2f2e1b359ee7de94defc4a15 con un total de ventas de $4,764.34 y 1 orden.
9. 3d979689f636322c62418b6346b1c6d2 con un total de ventas de $4,681.78 y 1 orden.
10. 1afc82cd60e303ef09b4ef9837c9505c con un total de ventas de $4,513.32 y 1 orden.

Estos clientes son los que más ventas han generado en la tienda en línea.


In [18]:
# Otra pregunta: tiempo de entrega y casos críticos
respuesta2 = ask_bi(db, "¿Cuál es el promedio de tiempo de entrega y hay entregas muy tardías?", provider="groq")
print(respuesta2)

El promedio de tiempo de entrega es de 12.50 días. Hay entregas muy tardías, con un máximo de 210 días. Los pedidos con entrega muy tardía son:

* order_id: ca07593549f1816d26a572e06dc1eab6, con 210 días de entrega
* order_id: 1b3190b2dfa9d789e1f14c05b647a14a, con 208 días de entrega
* order_id: 440d0d17af552815d15a9e41abe49359, con 196 días de entrega
* order_id: 2fb597c2f772eca01b1f5c561bf6cc7b, con 195 días de entrega
* order_id: 285ab9426d6982034523a855f55a885e, con 195 días de entrega

Es importante destacar que estos pedidos tienen un tiempo de entrega significativamente mayor que el promedio, lo que puede indicar problemas en el proceso de entrega o en la logística de la tienda.


## 6. Gráficas con DuckClaw (matplotlib y seaborn)

Puedes **generar gráficas** de dos formas:

1. **Directamente** con las funciones `plot_*` (datos desde DuckClaw, figuras con matplotlib/seaborn).
2. **En lenguaje natural** en el chatbot: pide *"muéstrame un gráfico de ventas por categoría"* o *"gráfica de los mejores vendedores"* y el agente guardará la imagen en la carpeta `output/`.

In [23]:
# Gráficas directas con DuckClaw (matplotlib/seaborn). Requiere: pip install matplotlib seaborn
from pathlib import Path
from duckclaw.bi import (
    plot_category_sales_bar,
    plot_top_sellers_bar,
    plot_review_score_pie,
    plot_delivery_days_histogram,
    plot_top_customers_bar,
)

OUT = Path("output")
OUT.mkdir(exist_ok=True)

# Generar y guardar gráficas
msg1 = plot_category_sales_bar(db, save_dir=str(OUT))
msg2 = plot_top_sellers_bar(db, save_dir=str(OUT), limit=8)
print(msg1)
print(msg2)

Gráfica guardada: /Users/juanjosearevalocamargo/Desktop/duckclaw/notebooks/output/ventas_por_categoria.png
Gráfica guardada: /Users/juanjosearevalocamargo/Desktop/duckclaw/notebooks/output/top_vendedores.png


In [20]:
# Mostrar las gráficas generadas en el notebook
from IPython.display import Image, display

for name in ["ventas_por_categoria.png", "top_vendedores.png"]:
    p = OUT / name
    if p.exists():
        display(Image(filename=str(p)))

### Gráfica desde lenguaje natural

Pide en texto que genere una gráfica; el agente usará las herramientas `plot_*` y te dirá en qué archivo se guardó.

In [21]:
# Ejemplo: pedir una gráfica en lenguaje natural (el agente guarda en output/ y te indica la ruta)
respuesta_grafica = ask_bi(db, "Muéstrame un gráfico de ventas por categoría de producto.", provider="groq")
print(respuesta_grafica)

El gráfico de ventas por categoría de producto se ha guardado en el archivo "output/category_sales_bar.png". Este gráfico muestra las ventas por categoría de producto, lo que puede ayudar a identificar las categorías más populares y tener una visión general de las ventas de la tienda online.


## 7. Ejemplos I/O en lenguaje natural

En esta sección todos los ejemplos están en formato **input → output** usando `ask_bi` (el agente llama funciones DuckClaw automáticamente).

In [ ]:
from duckclaw.bi import ask_bi

def demo_io(pregunta: str, provider: str = 'groq'):
    print(f'🟦 INPUT: {pregunta}')
    respuesta = ask_bi(db, pregunta, provider=provider)
    print(f'🟩 OUTPUT: {respuesta}')
    print('-' * 80)
    return respuesta


In [ ]:
# Ejemplos de negocio (texto)
preguntas_texto = [
    '¿Quiénes son los mejores vendedores este periodo?',
    '¿A qué clientes deberíamos fidelizar y por qué?',
    '¿Cuál es el tiempo promedio de entrega y hay casos críticos?',
    'Dame un resumen ejecutivo de ventas, ticket promedio y satisfacción.'
]

for p in preguntas_texto:
    demo_io(p)


In [ ]:
# Ejemplos de negocio (gráficas en lenguaje natural)
preguntas_graficas = [
    'Genera una gráfica de ventas por categoría de producto.',
    'Muéstrame un gráfico de top vendedores por ventas.',
    'Haz un histograma del tiempo de entrega.',
    'Crea una gráfica de distribución de reviews.'
]

for p in preguntas_graficas:
    demo_io(p)


### Nota
Cuando pidas gráficas por lenguaje natural, el agente devuelve la ruta del archivo generado (por ejemplo en `output/`). Si quieres visualizarlo en notebook, usa `IPython.display.Image`.

## 8. Chatbot con log + render automático de gráficas (VFS)

Esta sección agrega una experiencia tipo chat con trazabilidad persistente en **VFS local**:
- Guarda cada interacción en `vfs/bi_chat_memory.duckdb` (tabla `bi_chat_log`)
- Campos: `timestamp`, `input`, `output`, `image_path`, `provider`
- Si el agente devuelve una ruta de imagen (`.png/.jpg/.jpeg/.webp`), la muestra automáticamente en el notebook.

In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path

import duckclaw
from IPython.display import Image, display
from duckclaw.bi import ask_bi

# Persistencia en VFS local
VFS_DIR = REPO_ROOT / 'vfs'
VFS_DIR.mkdir(parents=True, exist_ok=True)
CHAT_DB_PATH = VFS_DIR / 'bi_chat_memory.duckdb'
chat_db = duckclaw.DuckClaw(str(CHAT_DB_PATH))

chat_db.execute("""
CREATE TABLE IF NOT EXISTS bi_chat_log (
    created_at TEXT,
    input_text TEXT,
    output_text TEXT,
    image_path TEXT,
    provider TEXT
)
""")

MAX_CHAT_ROWS = 1000
AUTO_PRUNE = True

print(f'🗄️ VFS chat memory: {CHAT_DB_PATH}')
print(f'🧹 Rotación automática: {AUTO_PRUNE} (máx {MAX_CHAT_ROWS} filas)')

CHAT_LOG = []

def _esc(s: str) -> str:
    return str(s).replace("'", "''")

def _extract_image_path(text: str):
    if not text:
        return None
    pattern = r'([\w\-./]+\.(?:png|jpg|jpeg|webp))'
    m = re.search(pattern, text, flags=re.IGNORECASE)
    if not m:
        return None
    raw = m.group(1)
    p = Path(raw)
    candidates = [p, Path.cwd() / p, Path.cwd().parent / p]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return None

def _query_json(sql: str):
    raw = chat_db.query(sql)
    return json.loads(raw) if isinstance(raw, str) else (raw or [])

def chat_log_count() -> int:
    rows = _query_json('SELECT COUNT(*) AS n FROM bi_chat_log')
    return int(rows[0]['n']) if rows else 0

def prune_chat_log(keep_last: int = MAX_CHAT_ROWS):
    keep_last = int(keep_last)
    if keep_last <= 0:
        return 0
    total = chat_log_count()
    if total <= keep_last:
        return 0
    rows = _query_json(
        f"SELECT created_at FROM bi_chat_log ORDER BY created_at DESC LIMIT 1 OFFSET {keep_last - 1}"
    )
    if not rows:
        return 0
    cutoff = rows[0]['created_at']
    before = total
    chat_db.execute(
        f"DELETE FROM bi_chat_log WHERE created_at < '{_esc(cutoff)}'"
    )
    after = chat_log_count()
    return max(before - after, 0)

def chat_log_stats():
    total = chat_log_count()
    provider_rows = _query_json(
        'SELECT provider, COUNT(*) AS n FROM bi_chat_log GROUP BY provider ORDER BY n DESC'
    )
    latest_rows = _query_json(
        'SELECT created_at FROM bi_chat_log ORDER BY created_at DESC LIMIT 1'
    )
    latest = latest_rows[0]['created_at'] if latest_rows else ''
    return {
        'total_rows': total,
        'providers': provider_rows,
        'latest_timestamp': latest,
    }

def _save_chat_record(question: str, answer: str, image_path: str, provider: str):
    ts = datetime.now().isoformat(timespec='seconds')
    chat_db.execute(
        'INSERT INTO bi_chat_log (created_at, input_text, output_text, image_path, provider) ' +
        f"VALUES ('{_esc(ts)}', '{_esc(question)}', '{_esc(answer)}', '{_esc(image_path)}', '{_esc(provider)}')"
    )
    if AUTO_PRUNE:
        prune_chat_log(MAX_CHAT_ROWS)

def load_chat_log(limit: int = 200):
    return _query_json(
        'SELECT created_at AS timestamp, input_text AS input, output_text AS output, image_path, provider ' +
        f"FROM bi_chat_log ORDER BY created_at DESC LIMIT {int(limit)}"
    )

def ask_bi_with_log(question: str, provider: str = 'groq'):
    answer = ask_bi(db, question, provider=provider)
    image = _extract_image_path(answer)
    image_str = str(image) if image else ''

    record = {
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'input': question,
        'output': answer,
        'image_path': image_str,
        'provider': provider,
    }
    CHAT_LOG.append(record)
    _save_chat_record(question, answer, image_str, provider)
    return answer, image


In [ ]:
# Chat interactivo con render automático de gráfica
print("🤖 BI Chat Pro listo. Escribe tu pregunta (o 'salir').")
while True:
    q = input("👤 Tú: ").strip()
    if not q:
        continue
    if q.lower() in ('salir', 'exit', 'quit', 'q'):
        print("👋 Fin de la sesión.")
        break

    ans, img = ask_bi_with_log(q, provider='groq')
    print("📊 BI:", ans)

    if img is not None:
        print(f"🖼️ Mostrando gráfica: {img}")
        display(Image(filename=str(img)))

    print('-' * 90)


In [ ]:
# Ver/exportar log de conversación desde VFS
import pandas as pd

stats = chat_log_stats()
print('📈 Stats:', stats)

log_df = pd.DataFrame(load_chat_log(limit=500))
display(log_df.head(20))

# Limpieza manual opcional
# deleted = prune_chat_log(keep_last=300)
# print(f'🧹 Filas eliminadas: {deleted}')

# Exportar snapshot opcional
# Path('output').mkdir(exist_ok=True)
# log_df.to_csv('output/chat_log_vfs.csv', index=False)


In [ ]:
# Mantenimiento rápido del VFS
print('Antes:', chat_log_stats())
deleted = prune_chat_log(keep_last=500)
print(f'🧹 Eliminadas: {deleted}')
print('Después:', chat_log_stats())
